# Lookzi — AI Virtual Try-On (Google Colab)

**Birinchi ishga tushirishdan oldin:** `Runtime → Change runtime type → T4 GPU`

| Qadam | Nima qiladi | 1-marta | Keyingi marta |
|-------|-------------|---------|---------------|
| 1 | Google Drive ulash | ~10 sek | ~10 sek |
| 2 | GPU tekshirish | - | - |
| 3 | Repo clone + paketlar o'rnatish | ~5-10 daq | ~1-2 daq (cache) |
| 4 | Model weights yuklash | ~10-15 daq | **0 sek (Drive'da)** |
| 5 | Ishga tushirish | - | - |

> **Muhim:** 3-qadamdan boshlab hamma narsa `Google Drive/Lookzi/` papkasiga saqlanadi.  
> Keyingi sessiyalarda weights qayta yuklanmaydi — Drive'dan olinadi.

In [ ]:
# ── Qadam 1: Google Drive ulash ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Drive'dagi asosiy papka
LOOKZI_DRIVE = '/content/drive/MyDrive/Lookzi'
PIP_CACHE    = f'{LOOKZI_DRIVE}/pip_cache'
WEIGHTS_DIR  = f'{LOOKZI_DRIVE}/weights'
HF_CACHE     = f'{LOOKZI_DRIVE}/hf_cache'

for d in [LOOKZI_DRIVE, PIP_CACHE, WEIGHTS_DIR, HF_CACHE]:
    os.makedirs(d, exist_ok=True)

print('Drive ulandi!')
print(f'Asosiy papka: {LOOKZI_DRIVE}')

# Drive'dagi papkalarni ko'rsatish
!ls -lh "{LOOKZI_DRIVE}"

In [ ]:
# ── Qadam 2: GPU tekshirish ───────────────────────────────────────────────
!nvidia-smi

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU      : {p.name}')
    print(f'VRAM     : {p.total_memory / 1e9:.1f} GB')
else:
    print()
    print('GPU topilmadi!')
    print('Runtime > Change runtime type > T4 GPU ni tanlang, keyin qayta ishga tushing.')

In [ ]:
# ── Qadam 3: Repo clone + paketlar o'rnatish ──────────────────────────────
# pip cache Drive'da saqlanadi — keyingi safar 5-10x tezroq bo'ladi
import os, sys

LOOKZI_DRIVE = '/content/drive/MyDrive/Lookzi'
PIP_CACHE    = f'{LOOKZI_DRIVE}/pip_cache'
REPO_DIR     = '/content/Lookzi-v0.1'

# Repo clone yoki yangilash
if os.path.exists(REPO_DIR):
    print('Repo mavjud — yangilanmoqda...')
    !git -C "{REPO_DIR}" pull --quiet
else:
    print('Repo yuklanmoqda...')
    !git clone https://github.com/Mohamed-Kudratov/Lookzi-v0.1.git "{REPO_DIR}" --quiet

os.chdir(REPO_DIR)
print(f'Repo: {os.getcwd()}')

# Paketlar o'rnatish (pip cache Drive'da — keyingi safar tez)
print('\nPaketlar o\'rnatilmoqda (1-marta 5-10 daqiqa, keyingi safar 1-2 daqiqa)...')
!pip install . --cache-dir="{PIP_CACHE}" -q
!pip install 'fastapi>=0.100' 'uvicorn[standard]' 'gradio>=6.0' --cache-dir="{PIP_CACHE}" -q

# Cache yangilash
import importlib
importlib.invalidate_caches()

# Tekshirish
import torch, gradio, fashn_vton
print(f'\nTorch   : {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'Gradio  : {gradio.__version__}')
print('fashn_vton: OK')
print('\nHamma paketlar tayyor!')

In [ ]:
# ── Qadam 4: Model weights (Drive'da saqlanadi, bir marta yuklanadi) ──────
import os

LOOKZI_DRIVE = '/content/drive/MyDrive/Lookzi'
WEIGHTS_DIR  = f'{LOOKZI_DRIVE}/weights'
HF_CACHE     = f'{LOOKZI_DRIVE}/hf_cache'
REPO_DIR     = '/content/Lookzi-v0.1'

# HuggingFace cache'ni Drive'ga yo'naltirish
os.environ['HF_HOME'] = HF_CACHE

os.chdir(REPO_DIR)

# Repo'dagi weights papkasini Drive'ga symlink qilish
weights_link = f'{REPO_DIR}/weights'
if os.path.islink(weights_link):
    os.unlink(weights_link)
    os.symlink(WEIGHTS_DIR, weights_link)
elif not os.path.exists(weights_link):
    os.symlink(WEIGHTS_DIR, weights_link)

# Weights mavjudligini tekshirish
model_file = f'{WEIGHTS_DIR}/model.safetensors'
yolox_file = f'{WEIGHTS_DIR}/dwpose/yolox_l.onnx'

if os.path.exists(model_file) and os.path.exists(yolox_file):
    size_gb = os.path.getsize(model_file) / 1e9
    print(f'Weights Drive\'da mavjud — qayta yuklanmaydi!')
    print(f'  model.safetensors : {size_gb:.2f} GB')
    !ls -lh "{WEIGHTS_DIR}/"
    !ls -lh "{WEIGHTS_DIR}/dwpose/"
else:
    print('Weights yuklanmoqda (~3 GB, faqat 1-marta)...')
    print('  - fashn-ai/fashn-vton-1.5  (model.safetensors)')
    print('  - fashn-ai/DWPose          (yolox_l.onnx, dw-ll_ucoco_384.onnx)')
    print('  - fashn-ai/fashn-human-parser')
    print()
    !python scripts/download_weights.py --weights-dir "{WEIGHTS_DIR}"
    print('\nWeights Drive\'ga saqlandi!')

In [ ]:
# ── Qadam 5: Lookzi ishga tushirish ───────────────────────────────────────
# Public share URL quyida paydo bo'ladi (72 soat ishlaydi)
import sys, os

LOOKZI_DRIVE = '/content/drive/MyDrive/Lookzi'
REPO_DIR     = '/content/Lookzi-v0.1'

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.environ.update({
    'HF_HUB_DISABLE_SYMLINKS_WARNING': '1',
    'TOKENIZERS_PARALLELISM'          : 'false',
    'HF_HOME'                         : f'{LOOKZI_DRIVE}/hf_cache',
})

from app import build_ui, get_pipeline

# Model yuklash (weights Drive'dan o'qiladi)
print('Model yuklanmoqda (30-60 sekund)...')
pipe = get_pipeline()
print(f'Model tayyor: {pipe.device}')

# UI ishga tushirish — public URL paydo bo'ladi
demo = build_ui()
demo.launch(
    share=True,       # *.gradio.live public URL
    debug=False,
    show_error=True,
)